# 대규모 언어 모델 실습: 멀티모달 대규모 언어 모델

도입: 이 섹션에서는 멀티모달 대규모 언어 모델의 주요 아키텍처와 구축 방법을 소개합니다.

> 대규모 언어 모델의 등장으로 고차원의 지능이 언어 모달리티에서 충분히 발현될 수 있음을 확인했습니다. 실세계를 더욱 충실하게 시뮬레이션할 수 있는 멀티모달 대규모 언어 모델은 어떻게 더 강력한 멀티모달 이해 및 생성 능력을 구현할 수 있을까요? 멀티모달 대규모 언어 모델이 AGI 달성에 도움이 될 수 있을까요?


대규모 모델 에이전트는 미래 운영체제로의 여정을 시작했습니다. 그러나 대규모 모델은 개방형 에이전트 시나리오에서 위험 위협을 인식할 수 있을까요?


## 이 튜토리얼의 목표

1. 멀티모달 대규모 언어 모델의 유형 이해
2. 멀티모달 대규모 언어 모델의 범용 기술 프레임워크 습득
3. 멀티모달 대규모 언어 모델의 구축, 훈련 및 추론 습득

## 1. 이론적 배경 지식



### 1.1 멀티모달 대규모 언어 모델의 유형 이해

- 현존하는 멀티모달 대규모 언어 모델의 기능 및 모달리티 지원 분류


> 멀티모달 대규모 언어 모델(MLLM)을 구축하기에 앞서, 이 분야의 연구자들은 하나의 공통된 합의, 즉 핵심 전제에 도달했습니다: 스케일링 법칙과 창발적 현상으로 인해 현재의 언어 기반 LLM은 이미 강력한 의미 이해 능력을 갖추고 있으며, 이는 언어가 지능을 담는 핵심 모달리티가 되었음을 의미합니다. 따라서 언어 지능은 멀티모달 지능의 허브로 간주됩니다. 이러한 이유로 거의 모든 MLLM은 언어 기반 LLM 위에 구축되며, LLM을 두뇌나 중앙 처리 장치와 유사한 핵심 의사결정 모듈로 활용합니다. 다시 말해, 추가적인 외부 비텍스트 모달리티 모듈이나 인코더를 추가함으로써 LLM에 멀티모달 인식/조작 능력을 부여합니다.

현존하는 MLLM을 모달리티 지원 및 기능 지원 상황에 따라 다양한 유형으로 분류합니다.

![mllm](assets/MLLM-summary.png)


- 전체 튜토리얼 자료 참고: [[Slides](https://github.com/Lordog/dive-into-llms/blob/main/documents/chapter6/mllms.pdf)] 


- 관련 서베이 논문
    - [A Survey on Multimodal Large Language Models, https://github.com/BradyFU/Awesome-Multimodal-Large-Language-Models, 2023](https://github.com/BradyFU/Awesome-Multimodal-Large-Language-Models)
    - [MM-LLMs: Recent Advances in MultiModal Large Language Models, 2023](https://arxiv.org/pdf/2401.13601)


### 1.2 멀티모달 대규모 언어 모델의 범용 기술 프레임워크 이해


- 아키텍처 1: LLM을 작업 스케줄러로 활용 (LLM as Task Scheduler)

> 현재 커뮤니티에는 두 가지 일반적인 MLLM 아키텍처가 존재합니다.
첫 번째는 "LLM을 이산적 스케줄러/컨트롤러로 활용"하는 아키텍처입니다. 아래 그림과 같이, LLM의 역할은 텍스트 신호를 수신하고 하위 모듈에 텍스트 명령을 전달하는 것입니다. 시스템 내부의 모든 메시지 전달은 LLM이 출력하는 순수 텍스트 명령을 매개로 수행됩니다. 서로 다른 기능 모듈 간에는 직접적인 상호작용이 없습니다.


![Architecture1](assets/Architecture1.png)

- 아키텍처 2: LLM을 시스템의 핵심 구성 요소로 활용 (LLM as Joint Part of System)

> 두 번째 아키텍처는 인코더-LLM-디코더 프레임워크입니다.
이것은 현재 가장 널리 사용되는 아키텍처이기도 합니다. 여기서 LLM의 역할은 멀티모달 정보를 인식하고, 인코더-LLM-디코더 구조 내에서 스스로 응답과 조작을 수행하는 것입니다. 따라서 이 아키텍처와 첫 번째 아키텍처의 핵심 차이점은: LLM이 시스템의 핵심 연결 부분으로서 외부로부터 멀티모달 정보를 직접 수신하고, 더 원활한 방식으로 디코더/생성기에 지시를 전달한다는 것입니다. 인코더-LLM-디코더 프레임워크에서는 아래 그림과 같이, 인코더가 여러 모달리티의 인코딩 신호를 처리하고, LLM이 핵심 의사결정자 역할을 하며, 디코더가 멀티모달 출력을 관리합니다.

![Architecture2](assets/Architecture2.png)






## 2. 범용 멀티모달 대규모 언어 모델 실습

> "임의 모달리티에서 임의 모달리티로"의 범용 멀티모달 대규모 언어 모델 구축 과정을 실습합니다.


### 2.1 범용 통합 "임의 모달리티 간 변환" 멀티모달 대규모 언어 모델: NExT-GPT

> 미래의 MLLM 연구는 점점 더 범용적인 generalist 방향으로 발전할 것이며, 가능한 한 많은 모달리티와 기능을 포함하게 될 것입니다. NExT-GPT는 이 분야에서 가장 선구적인 연구 중 하나로, "임의 모달리티에서 임의 모달리티로"의 MLLM 개념을 최초로 도입했습니다. 이 아키텍처는 강력한 기능을 구현하며, 미래 멀티모달 대규모 언어 모델의 연구 방향에 기반을 마련했습니다.

![NExT-GPT](assets/NExT-GPT-screen.png)


> 이번 강의의 멀티모달 대규모 언어 모델 코드 실습 부분에서는 NExT-GPT의 코드를 대상으로 심층적인 분석과 실습을 진행합니다.

[NExT-GPT 프로젝트 페이지](https://next-gpt.github.io/)

[NExT-GPT GitHub 코드 저장소](https://github.com/NExT-GPT/NExT-GPT)




### 2.2 코드 프레임워크 구조

```
├── figures
├── data
│   ├── T-X_pair_data  
│   │   ├── audiocap                      # 텍스트-오디오 쌍 데이터
│   │   │   ├── audios                    # 오디오 파일
│   │   │   └── audiocap.json             # 오디오 캡션
│   │   ├── cc3m                          # 텍스트-이미지 쌍 데이터
│   │   │   ├── images                    # 이미지 파일
│   │   │   └── cc3m.json                 # 이미지 캡션
│   │   └── webvid                        # 텍스트-비디오 쌍 데이터
│   │   │   ├── videos                    # 비디오 파일
│   │   │   └── webvid.json               # 비디오 캡션
│   ├── IT_data                           # 지시 미세조정 데이터
│   │   ├── T+X-T_data                    # 텍스트+[이미지/오디오/비디오] → 텍스트 지시 데이터
│   │   │   ├── alpaca                    # 텍스트 지시 데이터
│   │   │   ├── llava                     # 시각 지시 데이터
│   │   ├── T-T+X                         # 합성된 텍스트 → 텍스트+[이미지/오디오/비디오] 지시 데이터
│   │   └── MosIT                         # 모달리티 전환 지시 조정 데이터
├── code
│   ├── config
│   │   ├── base.yaml                     # 모델 설정
│   │   ├── stage_1.yaml                  # 인코더 측 정렬 훈련 설정
│   │   ├── stage_2.yaml                  # 디코더 측 정렬 훈련 설정
│   │   └── stage_3.yaml                  # 지시 조정 훈련 설정
│   ├── dsconfig
│   │   ├── stage_1.json                  # 인코더 측 정렬 훈련용 DeepSpeed 설정
│   │   ├── stage_2.json                  # 디코더 측 정렬 훈련용 DeepSpeed 설정
│   │   └── stage_3.json                  # 지시 조정 훈련용 DeepSpeed 설정
│   ├── datast
│   │   ├── base_dataset.py
│   │   ├── catalog.py                    # 데이터셋 카탈로그 정보
│   │   ├── cc3m_datast.py                # 텍스트-이미지 쌍 데이터셋 처리 및 로딩
│   │   ├── audiocap_datast.py            # 텍스트-오디오 쌍 데이터셋 처리 및 로딩
│   │   ├── webvid_dataset.py             # 텍스트-비디오 쌍 데이터셋 처리 및 로딩
│   │   ├── T+X-T_instruction_dataset.py  # 텍스트+X→텍스트 지시 데이터셋 처리 및 로딩
│   │   ├── T-T+X_instruction_dataset.py  # 텍스트→텍스트+X 지시 데이터셋 처리 및 로딩
│   │   └── concat_dataset.py             # 다중 데이터셋 처리 및 로딩
│   ├── model                     
│   │   ├── ImageBind                     # ImageBind 모델 코드
│   │   ├── common
│   │   ├── anyToImageVideoAudio.py       # 메인 모델 파일
│   │   ├── agent.py
│   │   ├── modeling_llama.py
│   │   ├── custom_ad.py                  # 오디오 디퓨전
│   │   ├── custom_sd.py                  # 이미지 디퓨전
│   │   ├── custom_vd.py                  # 비디오 디퓨전
│   │   ├── layers.py                     # 출력 투영 레이어
│   │   └── ...  
│   ├── scripts
│   │   ├── train.sh                      # NExT-GPT 훈련 스크립트
│   │   └── app.sh                        # 데모 배포 스크립트
│   ├── header.py
│   ├── process_embeddings.py             # 캡션 임베딩 사전 계산
│   ├── train.py                          # 훈련
│   ├── inference.py                      # 추론
│   ├── demo_app.py                       # Gradio 데모 배포
│   └── ...
├── ckpt                           
│   ├── delta_ckpt                        # 학습 가능한 NExT-GPT 파라미터
│   │   ├── nextgpt         
│   │   │   ├── 7b_tiva_v0                # 로그 파일 저장 디렉토리
│   │   │   │   ├── log                   # 로그
│   └── ...       
│   ├── pretrained_ckpt                   # 사전학습 모듈의 고정 파라미터
│   │   ├── imagebind_ckpt
│   │   │   ├──huge                       # 버전
│   │   │   │   └──imagebind_huge.pth
│   │   ├── vicuna_ckpt
│   │   │   ├── 7b_v0                     # 버전
│   │   │   │   ├── config.json
│   │   │   │   ├── pytorch_model-00001-of-00002.bin
│   │   │   │   ├── tokenizer.model
│   │   │   │   └── ...
├── LICENCE.md
├── README.md
└── requirements.txt
```


### 2.3 환경 설치

먼저 저장소를 클론하고 필요한 환경을 설치합니다. 다음 명령어를 실행하여 환경 설치를 완료할 수 있습니다:

```
conda env create -n nextgpt python=3.8

conda activate nextgpt

# CUDA 11.6
conda install pytorch==1.13.1 torchvision==0.14.1 torchaudio==0.13.1 pytorch-cuda=11.6 -c pytorch -c nvidia

git clone https://github.com/NExT-GPT/NExT-GPT.git
cd NExT-GPT

pip install -r requirements.txt
```




### 2.4 시스템 추론 실습


#### 2.4.1 사전학습된 NExT-GPT 모델 체크포인트 로딩

- **단계 1**: `고정 파라미터` 로딩. [NExT-GPT](https://github.com/NExT-GPT/NExT-GPT)는 다음의 기존 모델 또는 모듈을 기반으로 훈련되었습니다. 아래 설명에 따라 체크포인트를 준비하세요.
    - `ImageBind`는 통합 이미지/비디오/오디오 인코더입니다. [여기](https://dl.fbaipublicfiles.com/imagebind/imagebind_huge.pth)에서 `huge` 버전의 사전학습 체크포인트를 다운로드할 수 있습니다. 그런 다음 `imagebind_huge.pth` 파일을 [[./ckpt/pretrained_ckpt/imagebind_ckpt/huge]](ckpt/pretrained_ckpt/imagebind_ckpt/)에 배치합니다.
    - `Vicuna`: 먼저 [[여기]](ckpt/pretrained_ckpt/prepare_vicuna.md)의 설명에 따라 LLaMA를 준비합니다. 그런 다음 사전학습 모델을 [[./ckpt/pretrained_ckpt/vicuna_ckpt/]](ckpt/pretrained_ckpt/vicuna_ckpt/)에 배치합니다.
    - `Image Diffusion`은 이미지 생성에 사용됩니다. NExT-GPT는 `v1-5` 버전의 [Stable Diffusion](https://huggingface.co/runwayml/stable-diffusion-v1-5)을 사용합니다. (_코드에서 자동으로 다운로드됩니다_)
    - `Audio Diffusion`은 오디오 콘텐츠 생성에 사용됩니다. NExT-GPT는 `l-full` 버전의 [AudioLDM](https://github.com/haoheliu/AudioLDM)을 사용합니다. (_코드에서 자동으로 다운로드됩니다_)
    - `Video Diffusion`은 비디오 생성에 사용됩니다. `v2_576w` 버전의 [ZeroScope](https://huggingface.co/cerspense/zeroscope_v2_576w)를 사용합니다. (_코드에서 자동으로 다운로드됩니다_)

- **단계 2**: `학습 가능한 파라미터` 로딩.

NExT-GPT 시스템을 [[./ckpt/delta_ckpt/nextgpt/7b_tiva_v0]](./ckpt/delta_ckpt/nextgpt/7b_tiva_v0)에 배치합니다. 1) 직접 훈련한 파라미터를 사용하거나, 2) [HuggingFace](https://huggingface.co/ChocoWu/nextgpt_7b_tiva_v0)에서 사전학습된 체크포인트를 다운로드할 수 있습니다.


#### 2.4.2 Gradio 데모 배포

체크포인트 로딩이 완료되면, 다음과 같이 로컬에서 데모를 실행할 수 있습니다:
```angular2html
cd ./code
bash scripts/app.sh
```
주요 파라미터는 다음과 같습니다:
- `--nextgpt_ckpt_path`: 사전학습된 NExT-GPT 파라미터의 경로.



#### 2.4.3 테스트 사례 실습

현재 버전은 텍스트, 이미지, 비디오, 오디오 네 가지 모달리티의 임의 조합 입력을 지원하며, 임의 조합 모달리티의 출력이 가능합니다.
또한 다중 턴 컨텍스트 상호작용을 지원합니다.

직접 실행하여 테스트 결과를 확인해 보세요.

- **Case-1**: 입력 T+I, 출력 T+A

![case1](assets/T+I-T+A.png)


- **Case-2**: 입력 T+V, 출력 T+A

![case2](assets/T+V-T+A.png)


- **Case-3**: 입력 T+I, 출력 T+I+V

![case3](assets/T+I-T+I+V.png)


- **Case-4**: 입력 T, 출력 T+I+V+A

![case4](assets/T-T+I+V+A.png)







### 2.5 시스템 훈련 과정


#### 2.5.1 데이터 준비

모델 훈련을 위해 다음 데이터셋을 다운로드하세요:

A) T-X 쌍 데이터
  - ***텍스트-이미지*** 쌍의 `CC3M` 데이터는 이 설명을 참고하세요 [[here]](./data/T-X_pair_data/cc3m/prepare.md). 그런 다음 데이터를 [[./data/T-X_pair_data/cc3m]](./data/T-X_pair_data/cc3m)에 배치합니다.
  - ***텍스트-비디오*** 쌍의 `WebVid` 데이터는 [[instruction]](./data/T-X_pair_data/webvid/prepare.md)을 참고하세요. 파일은 [[./data/T-X_pair_data/webvid]](./data/T-X_pair_data/webvid)에 저장합니다.
  - ***텍스트-오디오*** 쌍의 `AudioCap` 데이터는 [[instruction]](./data/T-X_pair_data/audiocap/prepare.md)을 참고하세요. 데이터를 [[./data/T-X_pair_data/audiocap]](./data/T-X_pair_data/audiocap)에 저장합니다.

B) 지시 미세조정 데이터
  - T+X-T
    - ***시각 지시 데이터*** (`LLaVA` 출처)는 [여기](https://github.com/haotian-liu/LLaVA/blob/main/docs/Data.md)에서 다운로드하여 [[./data/IT_data/T+X-T_data/llava]](./data/IT_data/T+X-T_data/llava/)에 배치합니다.
    - ***텍스트 지시 데이터*** (`Alpaca` 출처)는 [여기](https://github.com/tatsu-lab/stanford_alpaca)에서 다운로드하여 [[./data/IT_data/T+X-T_data/alpaca/]](data/IT_data/T+X-T_data/alpaca/)에 배치합니다.
    - ***비디오 지시 데이터*** (`VideoChat` 출처)는 [여기](https://github.com/OpenGVLab/InternVideo/tree/main/Data/instruction_data)에서 다운로드하여 [[./data/IT_data/T+X-T_data/videochat/]](data/IT_data/T+X-T_data/videochat/)에 배치합니다.

    참고: 데이터셋 다운로드 후, `preprocess_dataset.py`를 실행하여 데이터셋을 전처리하고 형식을 통일합니다.
  - T-X+T (T2M)
    - `T-X+T` 지시 데이터셋(T2M)은 [[./data/IT_data/T-T+X_data]](./data/IT_data/T-T+X_data)에 저장되어 있습니다.
   
  - MosIT
    - [여기](./data/IT_data/MosIT_data/)에서 데이터 다운로드 설명을 확인하세요. 최종적으로 [[./data/IT_data/MosIT_data/]](./data/IT_data/MosIT_data/)에 배치합니다.




#### 2.5.2 임베딩 벡터 준비

NExT-GPT의 디코더 측 정렬 훈련에서는 Signal Token과 캡션의 표현 간 거리를 최소화합니다. 시스템의 효율성을 보장하고 시간 및 메모리 비용을 절약하기 위해, 각 디퓨전 모델의 텍스트 인코더를 사용하여 이미지, 오디오, 비디오 캡션의 텍스트 임베딩을 사전 계산합니다.

NExT-GPT 훈련을 시작하기 전에 다음 명령어를 실행하세요. 생성된 `embedding` 파일은 [[./data/embed]](./data/embed)에 저장됩니다.
```
cd ./code/
python process_embeddings.py ../data/T-X_pair_data/cc3m/cc3m.json image ../data/embed/ runwayml/stable-diffusion-v1-5
```


파라미터 설명:
- args[1]: 캡션 파일 경로;
- args[2]: 모달리티, `image`, `video`, `audio` 중 선택;
- args[3]: 임베딩 벡터 파일 저장 경로;
- args[4]: 해당 사전학습 디퓨전 모델 이름.




#### 2.5.3 3단계 훈련


먼저 기본 설정 파일 [[./code/config/base.yaml]](./code/config/base.yaml)을 참고하여 전체 모듈의 기본 시스템 설정을 확인합니다.

그런 다음 아래 스크립트를 실행하여 NExT-GPT 훈련을 시작합니다:
```
cd ./code
bash scripts/train.sh
```

명령어 예시:
```
deepspeed --include localhost:0 --master_addr 127.0.0.1 --master_port 28459 train.py \
    --model nextgpt \
    --stage 1\
    --save_path  ../ckpt/delta_ckpt/nextgpt/7b_tiva_v0/\
    --log_path ../ckpt/delta_ckpt/nextgpt/7b_tiva_v0/log/
```
주요 파라미터:
- `--include`: `localhost:0`은 DeepSpeed에서 GPU cuda 번호 `0`을 의미합니다.
- `--stage`: 훈련 단계.
- `--save_path`: 훈련된 delta 가중치를 저장할 디렉토리. 이 디렉토리는 자동으로 생성됩니다.
- `--log_path`: 로그 파일을 저장할 디렉토리.



NExT-GPT의 전체 훈련은 3단계로 구성됩니다:

- **단계 1**: 인코더 측 LLM 중심 멀티모달 정렬. 이 단계에서는 ***입력 투영 레이어***를 훈련하며, ImageBind, LLM, 출력 투영 레이어는 고정합니다.
  
  위의 `train.sh` 스크립트를 실행하고 다음과 같이 설정합니다: `--stage 1`
  
  실행 설정 파일 [[./code/config/stage_1.yaml]](./code/config/stage_1.yaml)과 DeepSpeed 설정 파일 [[./code/dsconfig/stage_1.yaml]](./code/dsconfig/stage_1.yaml)을 참고하여 더 자세한 단계별 설정 정보를 확인하세요.

  이 단계에서 사용되는 데이터셋은 `dataset_name_list`에 포함되어 있으며, 데이터셋 이름은 [[./code/dataset/catalog.py]](./code/dataset/catalog.py)의 정의와 정확히 일치해야 합니다.



- **단계 2**: 디코더 측 지시 따르기 정렬. 이 단계에서는 ***출력 투영 레이어***를 훈련하며, ImageBind, LLM, 입력 투영 레이어는 고정합니다.

  위의 `train.sh` 스크립트를 실행하고 다음과 같이 설정합니다: `--stage 2`

  실행 설정 파일 [[./code/config/stage_2.yaml]](./code/config/stage_2.yaml)과 DeepSpeed 설정 파일 [[./code/dsconfig/stage_2.yaml]](./code/dsconfig/stage_2.yaml)을 참고하여 더 자세한 단계별 설정 정보를 확인하세요.



- **단계 3**: 지시 조정. 이 단계에서는 지시 데이터셋에 대해 다음을 조정합니다: 1) LoRA를 통한 ***LLM*** 조정, 2) ***입력 투영 레이어*** 조정, 3) ***출력 투영 레이어*** 조정.

  위의 `train.sh` 스크립트를 실행하고 다음과 같이 설정합니다: `--stage 3`

  실행 설정 파일 [[./code/config/stage_3.yaml]](./code/config/stage_3.yaml)과 DeepSpeed 설정 파일 [[./code/dsconfig/stage_3.yaml]](./code/dsconfig/stage_3.yaml)을 참고하여 더 자세한 단계별 설정 정보를 확인하세요.